# 12 — Robustness Checks

**Goal:** Test whether the headline results (null alpha; sentiment predicts ROA and
sales growth) are robust to pre-committed alternative specifications.

**Inputs:**
- `data/panel_monthly.parquet`, `data/panel_yearly.parquet`, `data/factors.parquet`
- `data/long_short_returns.parquet`

**Output:** `output/table_robustness.csv`

**Pre-committed robustness specifications:**
1. **Review-count threshold:** restrict to firm-years with ≥30 reviews
2. **Precision weighting (H2b):** weight panel regressions by √n_reviews
3. **Subsample / regime split:** 2013–2019 vs 2020–2023 (addresses the 2021–22 reversal)
4. **Sub-rating tests:** re-run H2b using each of the five sentiment sub-scores
5. **Falsification:** confirm lagged sentiment does not predict *prior*-year performance

Specifications 1-3 were recorded in commit 9ed24f0 (2026-05-14), before the outcome data were retrieved (2026-05-15). The falsification and sub-rating specs were drafted with the battery but first entered version control together with their results (e7e493c, 2026-05-21) - pre-specified rather than pre-committed. The regime split is descriptive, not pre-committed (see thesis 3.5).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from scipy.stats import mstats
# --- NaN-safe winsorization fix (2026-06-19): mstats.winsorize silently clips
# missing values to the cap, fabricating firm-years. Preserve NaN instead so the
# regression's dropna handles them. ---
import pandas as _pd
_orig_winsorize = mstats.winsorize
def _winsorize_nan_safe(a, *args, **kwargs):
    s = _pd.Series(a).astype(float); out = s.copy(); m = s.notna()
    out[m] = _orig_winsorize(s[m].to_numpy(), *args, **kwargs)
    return out.to_numpy()
mstats.winsorize = _winsorize_nan_safe


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"
OUTPUT = Path.home() / "thesis" / "output"

panel_yearly = pd.read_parquet(DATA_PROCESSED / "panel_yearly.parquet")
panel_monthly = pd.read_parquet(DATA_PROCESSED / "panel_monthly.parquet")
factors = pd.read_parquet(DATA_PROCESSED / "factors.parquet")

# Winsorise outcomes (same as notebook 11, for consistency)
for var in ["roa", "operating_margin", "sales_growth"]:
    panel_yearly[f"{var}_w"] = mstats.winsorize(
        panel_yearly[var].astype(float), limits=[0.01, 0.01]
    )
panel_yearly["leverage_w"] = mstats.winsorize(panel_yearly["leverage"].astype(float), limits=[0.01, 0.01])

# Restrict to the 2013-2023 headline window AFTER winsorizing (caps from full panel). All robustness checks below run on this window.
panel_yearly = panel_yearly[(panel_yearly["fyear"] >= 2013) & (panel_yearly["fyear"] <= 2023)].copy()
panel_monthly = panel_monthly[(panel_monthly["year"] >= 2013) & (panel_monthly["year"] <= 2023)].copy()

print(f"Yearly panel: {len(panel_yearly):,} firm-years")
print(f"Monthly panel: {len(panel_monthly):,} firm-months")
print(f"Verify kernel: {__import__('sys').executable}")


Yearly panel: 4,962 firm-years
Monthly panel: 59,609 firm-months
Verify kernel: /opt/anaconda3/envs/thesis/bin/python


In [2]:
# Helper function to run the H2b panel regression and extract the sentiment coefficient
def run_h2b(data, outcome, sentiment_var="sentiment_overall_lag", label=""):
    d = data.set_index(["gvkey", "fyear"]).dropna(
        subset=[outcome, sentiment_var, "log_at", "leverage_w"]
    )
    formula = f"{outcome} ~ {sentiment_var} + log_at + leverage_w + EntityEffects + TimeEffects"
    res = PanelOLS.from_formula(formula, data=d, drop_absorbed=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    return {
        "spec": label,
        "outcome": outcome,
        "coef": res.params[sentiment_var],
        "t_stat": res.tstats[sentiment_var],
        "p_value": res.pvalues[sentiment_var],
        "n_obs": int(res.nobs),
    }

# Baseline (2013-2023 headline window)
print("=== Baseline (2013-2023) ===")
for outcome in ["roa_w", "operating_margin_w", "sales_growth_w"]:
    r = run_h2b(panel_yearly, outcome, label="baseline")
    print(f"  {outcome:20s} coef={r['coef']:.5f}  t={r['t_stat']:.2f}  p={r['p_value']:.4f}  N={r['n_obs']:,}")

# Threshold: ≥30 reviews
print("\n=== Threshold: firm-years with ≥30 reviews ===")
panel_30 = panel_yearly[panel_yearly["n_reviews_lag"] >= 30].copy()
print(f"  (retains {len(panel_30):,} of {len(panel_yearly):,} firm-years)")
for outcome in ["roa_w", "operating_margin_w", "sales_growth_w"]:
    r = run_h2b(panel_30, outcome, label="threshold_30")
    print(f"  {outcome:20s} coef={r['coef']:.5f}  t={r['t_stat']:.2f}  p={r['p_value']:.4f}  N={r['n_obs']:,}")

=== Baseline (2013-2023) ===
  roa_w                coef=0.00379  t=1.68  p=0.0931  N=4,956
  operating_margin_w   coef=0.00095  t=0.22  p=0.8258  N=4,725
  sales_growth_w       coef=0.02110  t=1.80  p=0.0726  N=4,954

=== Threshold: firm-years with ≥30 reviews ===
  (retains 4,049 of 4,962 firm-years)
  roa_w                coef=0.00115  t=0.23  p=0.8178  N=4,043
  operating_margin_w   coef=-0.00847  t=-1.22  p=0.2229  N=3,946
  sales_growth_w       coef=0.01763  t=1.19  p=0.2333  N=4,042


In [3]:
def run_h2b_weighted(data, outcome, sentiment_var="sentiment_overall_lag", label=""):
    d = data.set_index(["gvkey", "fyear"]).dropna(
        subset=[outcome, sentiment_var, "log_at", "leverage_w", "n_reviews_lag"]
    ).copy()
    d["weight"] = np.sqrt(d["n_reviews_lag"])
    formula = f"{outcome} ~ {sentiment_var} + log_at + leverage_w + EntityEffects + TimeEffects"
    res = PanelOLS.from_formula(formula, data=d, weights=d["weight"], drop_absorbed=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    return res.params[sentiment_var], res.tstats[sentiment_var], res.pvalues[sentiment_var], int(res.nobs)

print("=== Precision weighting (weights = sqrt(n_reviews)) ===")
for outcome in ["roa_w", "operating_margin_w", "sales_growth_w"]:
    coef, t, p, n = run_h2b_weighted(panel_yearly, outcome)
    sig = "YES" if p < 0.05 else "No"
    print(f"  {outcome:20s} coef={coef:.5f}  t={t:.2f}  p={p:.4f}  N={n:,}  sig={sig}")

=== Precision weighting (weights = sqrt(n_reviews)) ===
  roa_w                coef=0.00218  t=0.48  p=0.6311  N=4,956  sig=No


  operating_margin_w   coef=-0.00496  t=-0.85  p=0.3953  N=4,725  sig=No
  sales_growth_w       coef=0.02641  t=1.78  p=0.0748  N=4,954  sig=No


In [4]:
# H2a alpha on two subperiods, and H2b on two subperiods
ls = pd.read_parquet(DATA_PROCESSED / "long_short_returns.parquet")
ls["date"] = pd.to_datetime(ls["date"])
reg = ls.merge(factors[["date","mkt_rf","smb","hml","rmw","cma","mom"]], on="date", how="inner")

def alpha_subperiod(data, start, end, label):
    d = data[(data["date"].dt.year >= start) & (data["date"].dt.year <= end)]
    y = d["long_short"].astype("float64")
    X = sm.add_constant(d[["mkt_rf","smb","hml","rmw","cma","mom"]].astype("float64"))
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    a = res.params["const"]; t = res.tvalues["const"]; p = res.pvalues["const"]
    print(f"  {label:18s} alpha={a*100:.3f}%/mo (ann. {a*12*100:.2f}%)  t={t:.2f}  p={p:.3f}  N={int(res.nobs)}")

print("=== H2a: alpha by subperiod ===")
alpha_subperiod(reg, 2013, 2023, "Full 2013-2023")
alpha_subperiod(reg, 2013, 2019, "Pre 2013-2019")
alpha_subperiod(reg, 2020, 2023, "Post 2020-2023")

print("\n=== H2b: ROA and sales growth by subperiod ===")
for start, end, label in [(2013, 2019, "2013-2019"), (2020, 2023, "2020-2023")]:
    sub = panel_yearly[(panel_yearly["fyear"] >= start) & (panel_yearly["fyear"] <= end)]
    for outcome in ["roa_w", "sales_growth_w"]:
        r = run_h2b(sub, outcome, label=label)
        print(f"  {label}  {outcome:16s} coef={r['coef']:.5f}  t={r['t_stat']:.2f}  p={r['p_value']:.4f}  N={r['n_obs']:,}")

=== H2a: alpha by subperiod ===
  Full 2013-2023     alpha=0.119%/mo (ann. 1.43%)  t=1.37  p=0.170  N=132
  Pre 2013-2019      alpha=0.038%/mo (ann. 0.46%)  t=0.34  p=0.733  N=84
  Post 2020-2023     alpha=0.277%/mo (ann. 3.32%)  t=2.12  p=0.034  N=48

=== H2b: ROA and sales growth by subperiod ===
  2013-2019  roa_w            coef=0.00510  t=1.88  p=0.0608  N=3,071
  2013-2019  sales_growth_w   coef=0.02911  t=2.29  p=0.0220  N=3,070
  2020-2023  roa_w            coef=-0.00013  t=-0.02  p=0.9817  N=1,885
  2020-2023  sales_growth_w   coef=0.01521  t=0.47  p=0.6384  N=1,884


In [5]:
print("=== H2b using each sentiment sub-rating (2013-2023) ===")
subratings = {
    "sentiment_wlb_lag": "Work-life balance",
    "sentiment_comp_lag": "Compensation",
    "sentiment_career_lag": "Career opportunities",
    "sentiment_culture_lag": "Culture",
    "sentiment_leadership_lag": "Senior leadership",
}

for outcome in ["roa_w", "sales_growth_w"]:
    print(f"\n  Outcome: {outcome}")
    for var, label in subratings.items():
        try:
            r = run_h2b(panel_yearly, outcome, sentiment_var=var, label=label)
            sig = "*" if r["p_value"] < 0.05 else " "
            print(f"    {label:22s} coef={r['coef']:.5f}  t={r['t_stat']:.2f}  p={r['p_value']:.4f} {sig}")
        except Exception as e:
            print(f"    {label:22s} FAILED: {str(e)[:60]}")

=== H2b using each sentiment sub-rating (2013-2023) ===

  Outcome: roa_w


    Work-life balance      coef=0.00142  t=0.62  p=0.5322  
    Compensation           coef=0.00371  t=1.20  p=0.2295  
    Career opportunities   coef=0.00266  t=1.05  p=0.2930  


    Culture                coef=0.00246  t=1.11  p=0.2692  
    Senior leadership      coef=0.00404  t=1.74  p=0.0815  

  Outcome: sales_growth_w


    Work-life balance      coef=0.02249  t=2.15  p=0.0317 *
    Compensation           coef=0.03196  t=2.39  p=0.0170 *
    Career opportunities   coef=0.00983  t=0.97  p=0.3338  
    Culture                coef=0.02254  t=2.37  p=0.0181 *


    Senior leadership      coef=0.02290  t=2.36  p=0.0183 *


In [6]:
# Falsification: does sentiment predict PRIOR-year performance?
# If our lag structure captures a real predictive relationship, then sentiment
# (year t-1) should predict performance in year t (forward) but NOT performance
# in year t-2 (backward). A significant "backward" result would suggest the
# relationship is spurious / driven by a persistent confounder.

# Build a backward outcome: performance in the year BEFORE the sentiment year.
# panel_yearly has sentiment_overall_lag (from t-1) aligned to outcome year t.
# We create a "lead" outcome = next year's outcome, which relative to sentiment
# is actually testing whether t-1 sentiment predicts t-2 performance.

panel_falsify = panel_yearly.sort_values(["gvkey", "fyear"]).copy()

# Future ROA (year t+1) — sentiment from t-1 should NOT predict this any better
# than it predicts t, if anything. More direct test: does t-1 sentiment predict
# t-2 ROA (backward)? Construct lagged outcome.
# FIX 2026-07-12: positional groupby().shift(2) replaced with an explicit
# fiscal-year merge. panel_yearly has membership gaps (firms leave and re-enter
# the index), so a two-ROW shift grabbed t-3-or-older outcomes for some
# firm-years; the merge takes the outcome from exactly fyear - 2.
_bk = panel_falsify[["gvkey", "fyear", "roa_w", "sales_growth_w"]].copy()
_bk["fyear"] = _bk["fyear"] + 2
panel_falsify = panel_falsify.merge(
    _bk.rename(columns={"roa_w": "roa_w_backward",
                        "sales_growth_w": "sales_growth_w_backward"}),
    on=["gvkey", "fyear"], how="left", validate="m:1")

print("=== Falsification: does t-1 sentiment predict t-2 (PRIOR) performance? ===")
print("    (a significant result here would be a red flag)\n")
for outcome in ["roa_w_backward", "sales_growth_w_backward"]:
    try:
        r = run_h2b(panel_falsify, outcome, label="falsification")
        sig = "RED FLAG" if r["p_value"] < 0.05 else "OK (insignificant)"
        print(f"  {outcome:24s} coef={r['coef']:.5f}  t={r['t_stat']:.2f}  p={r['p_value']:.4f}  [{sig}]")
    except Exception as e:
        print(f"  {outcome:24s} FAILED: {str(e)[:60]}")

=== Falsification: does t-1 sentiment predict t-2 (PRIOR) performance? ===
    (a significant result here would be a red flag)

  roa_w_backward           coef=0.00754  t=2.13  p=0.0332  [RED FLAG]
  sales_growth_w_backward  coef=-0.01065  t=-0.98  p=0.3285  [OK (insignificant)]


In [7]:
# Assemble the full robustness battery (Table 5) into one DataFrame and write
# output/table_robustness.csv, so the chapter's robustness numbers regenerate
# from committed code rather than from hand-transcribed stdout.
_LBL = {"roa_w": "Return on assets", "sales_growth_w": "Sales growth"}
rob_rows = []

def _record(spec, outcome, coef, t, p, n):
    rob_rows.append({"specification": spec, "outcome": _LBL[outcome],
                     "coef": coef, "t_stat": t, "p_value": p, "n_obs": n})

# Headline
for oc in ["sales_growth_w", "roa_w"]:
    r = run_h2b(panel_yearly, oc, label="headline")
    _record("Headline", oc, r["coef"], r["t_stat"], r["p_value"], r["n_obs"])

# >=30 reviews
panel_30 = panel_yearly[panel_yearly["n_reviews_lag"] >= 30].copy()
for oc in ["sales_growth_w", "roa_w"]:
    r = run_h2b(panel_30, oc, label="threshold_30")
    _record(">=30 reviews", oc, r["coef"], r["t_stat"], r["p_value"], r["n_obs"])

# Precision weighting (sqrt n_reviews)
for oc in ["sales_growth_w", "roa_w"]:
    coef, t, p, n = run_h2b_weighted(panel_yearly, oc)
    _record("Precision-weighted (sqrt n)", oc, coef, t, p, n)

# Falsification: does t-1 sentiment predict t-2 (prior) performance?
panel_fal = panel_yearly.sort_values(["gvkey", "fyear"]).copy()
# FIX 2026-07-12: fiscal-year merge instead of positional shift(2) — see cell above.
_bk2 = panel_fal[["gvkey", "fyear", "roa_w", "sales_growth_w"]].copy()
_bk2["fyear"] = _bk2["fyear"] + 2
panel_fal = panel_fal.merge(
    _bk2.rename(columns={"roa_w": "roa_w_backward",
                         "sales_growth_w": "sales_growth_w_backward"}),
    on=["gvkey", "fyear"], how="left", validate="m:1")
for oc, bk in [("sales_growth_w", "sales_growth_w_backward"), ("roa_w", "roa_w_backward")]:
    r = run_h2b(panel_fal, bk, label="falsification")
    _record("Falsification (backward t-2)", oc, r["coef"], r["t_stat"], r["p_value"], r["n_obs"])

# Regime sub-periods
for start, end in [(2013, 2019), (2020, 2023)]:
    sub = panel_yearly[(panel_yearly["fyear"] >= start) & (panel_yearly["fyear"] <= end)].copy()
    for oc in ["sales_growth_w", "roa_w"]:
        r = run_h2b(sub, oc, label=f"{start}-{end}")
        _record(f"{start}-{end} sub-period", oc, r["coef"], r["t_stat"], r["p_value"], r["n_obs"])

table_robustness = pd.DataFrame(rob_rows)
for _col, _nd in [("coef", 5), ("t_stat", 3), ("p_value", 4)]:
    table_robustness[_col] = table_robustness[_col].round(_nd)
print(table_robustness.to_string(index=False))
table_robustness.to_csv(OUTPUT / "table_robustness.csv", index=False)
print(f"\nSaved {len(table_robustness)} robustness rows to {OUTPUT / 'table_robustness.csv'}")

               specification          outcome     coef  t_stat  p_value  n_obs
                    Headline     Sales growth  0.02110   1.796   0.0726   4954
                    Headline Return on assets  0.00379   1.680   0.0931   4956
                >=30 reviews     Sales growth  0.01763   1.192   0.2333   4042
                >=30 reviews Return on assets  0.00115   0.230   0.8178   4043
 Precision-weighted (sqrt n)     Sales growth  0.02641   1.782   0.0748   4954
 Precision-weighted (sqrt n) Return on assets  0.00218   0.480   0.6311   4956
Falsification (backward t-2)     Sales growth -0.01065  -0.977   0.3285   3724
Falsification (backward t-2) Return on assets  0.00754   2.131   0.0332   3725
        2013-2019 sub-period     Sales growth  0.02911   2.291   0.0220   3070
        2013-2019 sub-period Return on assets  0.00510   1.876   0.0608   3071
        2020-2023 sub-period     Sales growth  0.01521   0.470   0.6384   1884
        2020-2023 sub-period Return on assets -0.000

In [8]:
print("=" * 60)
print("Headline results restricted to 2013–2023 (per JJ)")
print("=" * 60)

# H2a alpha: 2013–2023
ls_check = pd.read_parquet(DATA_PROCESSED / "long_short_returns.parquet")
ls_check["date"] = pd.to_datetime(ls_check["date"])
factors_check = pd.read_parquet(DATA_PROCESSED / "factors.parquet")
reg_check = ls_check.merge(
    factors_check[["date","mkt_rf","smb","hml","rmw","cma","mom"]],
    on="date", how="inner"
)
sub = reg_check[(reg_check["date"].dt.year >= 2013) & (reg_check["date"].dt.year <= 2023)]
y = sub["long_short"].astype("float64")
X = sm.add_constant(sub[["mkt_rf","smb","hml","rmw","cma","mom"]].astype("float64"))
res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
print(f"\nH2a alpha (2013–2023):")
print(f"  Monthly: {res.params['const']*100:.4f}%  Annualized: {res.params['const']*12*100:.2f}%")
print(f"  t-stat: {res.tvalues['const']:.2f}  p-value: {res.pvalues['const']:.3f}  N months: {int(res.nobs)}")
print(f"  SMB loading: {res.params['smb']:.3f}  t: {res.tvalues['smb']:.2f}")

# H2b: 2013–2023
print(f"\nH2b panel regressions (2013–2023):")
py23 = panel_yearly[(panel_yearly["fyear"] >= 2013) & (panel_yearly["fyear"] <= 2023)]
for outcome in ["roa_w", "operating_margin_w", "sales_growth_w"]:
    r = run_h2b(py23, outcome, label="2013-2023")
    sig = "YES" if r["p_value"] < 0.05 else "No"
    print(f"  {outcome:20s} coef={r['coef']:.5f}  t={r['t_stat']:.2f}  p={r['p_value']:.4f}  N={r['n_obs']:,}  sig={sig}")

# Regime split within 2013–2023
print(f"\nRegime split within 2013–2023:")
alpha_subperiod(reg_check, 2013, 2019, "Pre 2013-2019")
alpha_subperiod(reg_check, 2020, 2023, "Post 2020-2023")  # shorter post-period

Headline results restricted to 2013–2023 (per JJ)

H2a alpha (2013–2023):
  Monthly: 0.1188%  Annualized: 1.43%
  t-stat: 1.37  p-value: 0.170  N months: 132
  SMB loading: -0.225  t: -3.59

H2b panel regressions (2013–2023):
  roa_w                coef=0.00379  t=1.68  p=0.0931  N=4,956  sig=No
  operating_margin_w   coef=0.00095  t=0.22  p=0.8258  N=4,725  sig=No
  sales_growth_w       coef=0.02110  t=1.80  p=0.0726  N=4,954  sig=No

Regime split within 2013–2023:
  Pre 2013-2019      alpha=0.038%/mo (ann. 0.46%)  t=0.34  p=0.733  N=84
  Post 2020-2023     alpha=0.277%/mo (ann. 3.32%)  t=2.12  p=0.034  N=48
